In [1]:
# ================================
# 1. LIBRARIES IMPORT
# ================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ================================
# 2. LOADING THE DATASET
# ================================
import os
from google.colab import files

file_name = "Food_tweets.csv"
if not os.path.exists(file_name):
    print(f"File '{file_name}' not found. Please upload it.")
    uploaded = files.upload()
    if file_name not in uploaded:
        raise FileNotFoundError(f"'{file_name}' was not uploaded. Please try again.")

df = pd.read_csv(file_name, engine='python', on_bad_lines='skip')

# ================================
# 3. DATA CLEANING
# ================================

df = df.dropna(subset=[
    'tweet.text',
    'tweet.public_metrics.like_count',
    'tweet.public_metrics.retweet_count',
    'tweet.public_metrics.reply_count',
    'tweet.public_metrics.quote_count',
    'user.public_metrics.followers_count'
])

# KEEP ONLY TWEETS IN ENGLISH
df = df[df['tweet.lang'] == 'en']

# ================================
# 4. TEXTE CLEANING
# ================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_text"] = df["tweet.text"].apply(clean_text)

# ================================
# 5. SENTIMENT ANALYSIS
# ================================

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df["sentiment"] = df["clean_text"].apply(get_sentiment)

# ================================
# 6. DEFINITION OF POPULARITY
# ================================

df["popularity"] = (
    df["tweet.public_metrics.like_count"] +
    df["tweet.public_metrics.retweet_count"] +
    df["tweet.public_metrics.reply_count"] +
    df["tweet.public_metrics.quote_count"] +
    df["sentiment"]
)

# ================================
# 7. VISUALISATIONS (FIGURES)
# ================================

# Figure 1 : Distribution of likes
plt.figure()
plt.hist(df["tweet.public_metrics.like_count"], bins=50)
plt.title("Distribution of likes")
plt.xlabel("Likes")
plt.ylabel("Frequency")
plt.show()

# Figure 2 : Followers vs PopularitY
plt.figure()
plt.scatter(
    df["user.public_metrics.followers_count"],
    df["popularity"],
    alpha=0.3
)
plt.title("Followers vs Popularity")
plt.xlabel("Followers")
plt.ylabel("Popularity")
plt.show()

# Figure 3 : Sentiment distribution
plt.figure()
plt.hist(df["sentiment"], bins=50)
plt.title("Sentiment distribution")
plt.xlabel("Sentiment score")
plt.ylabel("Frequency")
plt.show()

# ================================
# 8. TF-IDF
# ================================

tfidf = TfidfVectorizer(max_features=500)
X_text = tfidf.fit_transform(df["clean_text"])

# Numeric variables
X_num = df[[
    "user.public_metrics.followers_count",
    "sentiment"
]].values

from scipy.sparse import hstack
X = hstack([X_text, X_num])

y = df["popularity"]

# ================================
# 9. SPLIT TRAIN / TEST
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 10. MODELS
# ================================

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100)
}

# ================================
# 11. TRAINING AND EVALUATION
# ================================

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append((name, rmse, r2))

    print(f"\n{name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 score: {r2:.3f}")

# ================================
# 12. COMPARISON OF MODELS
# ================================

results_df = pd.DataFrame(results, columns=["Model", "RMSE", "R2"])
print("\nComparison of modEls :")
print(results_df)

ModuleNotFoundError: No module named 'textblob'